# Ringkasan evaluasi summarization per model

In [1]:
import json
import os
from pathlib import Path
from urllib.request import urlopen

import mlflow
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Jalankan Jupyter dari root proyek atau subfoldernya.")
load_dotenv(PROJECT_ROOT / ".env")

True

## Sumber dan filter

In [2]:
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = "summarization-quality-benchmark"
SPEED_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "summarization-speed-benchmark")
COMPLETE_ONLY = True
QUALITY_FILTERS = {
    "benchmark_group": "summeval-v1",
    "protocol_sha256": None,
    "metric_id": None,
    "config_sha256": None,
}
SPEED_FILTERS = {
    "benchmark_group": "walden-v1",
    "dataset_sha256": None,
    "prompt_sha256": None,
    "config_sha256": None,
    "testing": None,
}

mlflow.set_tracking_uri(TRACKING_URI)


In [3]:
def load_runs(experiment, run_filter, columns, filters):
    if mlflow.get_experiment_by_name(experiment) is None:
        raise RuntimeError(f"Eksperimen {experiment!r} tidak ditemukan di {TRACKING_URI}.")
    columns = {"run_id": "run_id", "start_time": "start_time", **columns}
    runs = mlflow.search_runs(
        experiment_names=[experiment],
        filter_string=run_filter,
        order_by=["attributes.start_time DESC", "attributes.run_id ASC"],
        output_format="pandas",
    ).reindex(columns=list(columns)).rename(columns=columns)
    for source, column in columns.items():
        if source.startswith("metrics."):
            runs[column] = pd.to_numeric(runs[column], errors="coerce")
    for column, value in filters.items():
        if value is not None:
            values = runs[column]
            if column == "testing":
                values, value = values.astype("string").str.lower(), str(value).lower()
            runs = runs.loc[values.eq(value)]
    return runs.reset_index(drop=True)


def latest_runs(runs, keys, required):
    return (
        runs.dropna(subset=[*keys, *required])
        .sort_values(["start_time", "run_id"], ascending=[False, True])
        .drop_duplicates(keys)
    )


In [4]:
evaluation_df = load_runs(EXPERIMENT_NAME, "tags.run_type = 'evaluation'", {
    "params.model": "model",
    "tags.metric_name": "metric",
    "metrics.score_mean": "score_mean",
    "metrics.score_median": "score_median",
    "metrics.coverage": "coverage",
    "status": "status",
    "tags.benchmark_group": "benchmark_group",
    "tags.protocol_sha256": "protocol_sha256",
    "tags.metric_id": "metric_id",
    "tags.config_sha256": "config_sha256",
}, QUALITY_FILTERS)

speed_metrics = [
    "summarization_minutes", "prefill_tokens_per_second", "generation_tokens_per_second",
]
speed_df = load_runs(
    SPEED_EXPERIMENT_NAME, "tags.run_type = 'benchmark' AND attributes.status = 'FINISHED'",
    {
        "params.model": "model",
        **{f"metrics.{metric}": metric for metric in speed_metrics},
        "tags.benchmark_group": "benchmark_group",
        "params.dataset_sha256": "dataset_sha256",
        "params.prompt_sha256": "prompt_sha256",
        "tags.config_sha256": "config_sha256",
        "params.testing": "testing",
    }, SPEED_FILTERS,
)


## Pilih run terbaru per model


In [5]:
model_speed_df = (
    latest_runs(speed_df, ["model"], ["summarization_minutes"])
    .sort_values(["summarization_minutes", "model"], kind="stable")
    .set_index("model")[speed_metrics]
)

In [6]:
eligible_df = evaluation_df
if COMPLETE_ONLY:
    eligible_df = eligible_df.loc[
        eligible_df["status"].eq("FINISHED") & eligible_df["coverage"].eq(1.0)
    ]
selected_runs_df = latest_runs(
    eligible_df, ["model", "metric"], ["score_mean", "score_median"],
)
statistics = ["score_mean", "score_median"]
model_metrics_df = selected_runs_df.pivot(
    index="model", columns="metric", values=statistics,
).swaplevel(0, 1, axis="columns")
available_metrics = set(selected_runs_df["metric"])
metric_priority = ["hhem_consistency", "bertscore_f1"]
metric_order = [metric for metric in metric_priority if metric in available_metrics]
metric_order += sorted(available_metrics.difference(metric_priority))
model_metrics_df = model_metrics_df.reindex(
    columns=pd.MultiIndex.from_product(
        [metric_order, statistics], names=["metric", "statistic"],
    ),
)
if metric_order:
    model_metrics_df = model_metrics_df.sort_values(
        by=[(metric, "score_mean") for metric in metric_order],
        ascending=False, na_position="last", kind="stable",
    )


## Speed dan kualitas per model

In [7]:
speed_display_df = model_speed_df.rename(columns={
    "summarization_minutes": "Summarization Time",
    "prefill_tokens_per_second": "Prefill TPS",
    "generation_tokens_per_second": "Generation TPS",
})
speed_display_df.columns = pd.MultiIndex.from_product(
    [["Speed"], speed_display_df.columns], names=["Metric", "Statistic"],
)
quality_display_df = model_metrics_df.rename(columns={
    "hhem_consistency": "HHEM Consistency",
    "bertscore_f1": "BERTScore F1 Semantic Quality",
    "score_mean": "Mean",
    "score_median": "Median",
}).rename_axis(columns=["Metric", "Statistic"])
combined_df = pd.concat([speed_display_df, quality_display_df], axis=1).rename_axis("Model")
combined_df = combined_df.loc[~combined_df.index.astype(str).str.contains("luna", case=False, na=False)]

def canonical_ollama_name(name):
    return name if ":" in name.rsplit("/", 1)[-1] else f"{name}:latest"

ollama_host = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/")
ollama_models = {}
try:
    with urlopen(f"{ollama_host}/api/tags", timeout=5) as response:
        installed_models = json.load(response)["models"]
    ollama_models = {
        canonical_ollama_name(model["name"]): model
        for model in installed_models
    }
except (OSError, ValueError) as error:
    print(f"Metadata model tidak tersedia dari Ollama: {error}")

model_metadata = [
    ollama_models.get(canonical_ollama_name(str(name)), {}) for name in combined_df.index
]
combined_df.insert(
    0,
    ("Resource", "Model Size"),
    [model["size"] / 1_000_000_000 if "size" in model else None for model in model_metadata],
)

model_display_names = {
    "gemma4:e2b": "Gemma 4 E2B",
    "gemma4:e4b-it-q4_K_M": "Gemma 4 E4B IT",
    "maternion/ling-3.0-tiny:8b-Q4_K_M": "Ling 3.0 Tiny 8B",
    "LiquidAI/lfm2.5-2.6b:q4_k_m": "LFM 2.5 2.6B",
    "openbmb/minicpm5-2b": "MiniCPM 5 2B",
    "qwen3.5:4b-q4_K_M": "Qwen 3.5 4B",
    "granite4.2:3b-q4_K_M": "Granite 4.2 3B",
}
model_display_names = {
    canonical_ollama_name(name): label for name, label in model_display_names.items()
}
display_df = combined_df.rename(index=lambda name: model_display_names.get(
    canonical_ollama_name(str(name)), name,
))

formats = {
    ("Resource", "Model Size"): "{:.2f} GB",
    ("Speed", "Summarization Time"): "{:.1f} minutes",
    ("Speed", "Prefill TPS"): "{:.0f} tok/sec",
    ("Speed", "Generation TPS"): "{:.0f} tok/sec",
}
formats.update({column: "{:.2%}" for column in quality_display_df.columns})
ranked_display_df = display_df.copy()
ranked_display_df.index = pd.MultiIndex.from_arrays(
    [range(1, len(display_df) + 1), display_df.index], names=["Rank", "Model"],
)

display(ranked_display_df.style.format(formats, na_rep="N/A"))

**Konfigurasi:**

- Semua model menggunakan quantization `Q4_K_M`.
- Temperature `0`, reasoning `False`, context window 65.536 token, maksimal output 8.192 token. 
- Tambahan top-k: `50` dan repeat penalty: `1,1` untuk summarization quality. 
- Perangkat: CPU untuk summarization atau Ollama, GPU (`cuda`) hanya untuk penilaian kualitas.
- **Summarization speed:** seluruh buku [Walden](https://www.gutenberg.org/cache/epub/205/pg205-images.html). Summarization dilakukan secara hierarkis: setiap bab dirangkum, lalu seluruh rangkuman disatukan. 
- **Summarization quality:** seluruh artikel split `test` pada [mteb/summeval](https://huggingface.co/datasets/mteb/summeval), rangkuman bahasa Inggris 2 sampai 3 kalimat, 
- Spesifiksi device benchmark: 16 GB RAM DDR5, 8 Core AMD CPU


In [8]:
for name, metadata in zip(display_df.index, model_metadata):
    quantization = metadata.get("details", {}).get("quantization_level", "N/A")
    print(f"{name}: {quantization}")

Gemma 4 E2B: Q4_K_M
Ling 3.0 Tiny 8B: Q4_K_M
LFM 2.5 2.6B: Q4_K_M
MiniCPM 5 2B: Q4_K_M
Gemma 4 E4B IT: Q4_K_M
Qwen 3.5 4B: Q4_K_M
Granite 4.2 3B: Q4_K_M


**Keterangan:**

Arti setiap metrik serta cara membacanya dijelaskan berikut:

- **Model Size** = Ukuran file model yang terpasang di Ollama, dalam GB. Angka ini menunjukkan kebutuhan penyimpanan.
- **Summarization Time** = Total waktu merangkum semua bab dan membuat rangkuman akhir, dalam menit. Lebih kecil berarti lebih cepat.
- **Prefill TPS** = Kecepatan model membaca dan memproses token input sebelum menghasilkan jawaban, dalam token/detik. Lebih tinggi berarti input diproses lebih cepat.
- **Generation TPS** = Kecepatan model menghasilkan token jawaban, dalam token/detik. Lebih tinggi berarti jawaban dihasilkan lebih cepat.
- **HHEM Consistency** = Konsistensi fakta rangkuman terhadap teks sumber, dinilai oleh [Vectara HHEM-2.1-Open](https://huggingface.co/vectara/hallucination_evaluation_model). Lebih tinggi berarti rangkuman lebih didukung oleh sumber.
- **BERTScore F1** = Kemiripan makna rangkuman model dengan rangkuman referensi manusia, menggunakan [BERTScore](https://github.com/Tiiiger/bert_score). Lebih tinggi berarti lebih mirip dengan referensi.
- **Mean** = Rata-rata skor seluruh artikel dalam run yang dipilih. Mempertimbangkan setiap skor, tetapi lebih mudah terpengaruh nilai ekstrem.
- **Median** = Nilai tengah skor artikel dalam run yang dipilih. Lebih tahan terhadap nilai ekstrem dan menggambarkan skor artikel yang tipikal.

Skor kualitas dinormalisasi ke skala 0-100%, misalnya 0,85 menjadi 85,00%. 


In [9]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

chart_df = pd.DataFrame({
    "Menit": display_df[("Speed", "Summarization Time")],
    "HHEM": display_df[("HHEM Consistency", "Mean")],
    "BERTScore": display_df[("BERTScore F1 Semantic Quality", "Mean")],
}).rename_axis("Model").reset_index()

# 1. Speed
speed = chart_df.dropna(subset=["Menit"]).sort_values("Menit")

fig_speed = px.bar(
    speed,
    x="Menit", y="Model", orientation="h",
    text_auto=".1f",
    title="Speed · waktu lebih kecil lebih baik",
    labels={"Menit": "Total waktu summarization (menit)"},
    template="plotly_white",
)
fig_speed.update_yaxes(autorange="reversed", title=None)
fig_speed.show()

# 2. Prefill dan generation: skala X terpisah agar keduanya terbaca.
tps = (
    display_df["Speed"][["Prefill TPS", "Generation TPS"]]
    .dropna(how="all")
    .sort_values("Generation TPS", ascending=False, na_position="last")
)
fig_tps = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    subplot_titles=("Prefill TPS", "Generation TPS"),
    horizontal_spacing=0.10,
)
for col, (metric, color) in enumerate([
    ("Prefill TPS", "#2563eb"),
    ("Generation TPS", "#0d9488"),
], start=1):
    values = tps[metric].dropna()
    fig_tps.add_trace(go.Bar(
        x=values, y=values.index, orientation="h",
        marker_color=color,
        texttemplate="%{x:.1f}", textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "%{y}<br>" + metric + ": %{x:.2f} tok/sec<extra></extra>"
        ),
        showlegend=False,
    ), row=1, col=col)
    upper = max(1, float(values.max()) * 1.25) if not values.empty else 1
    fig_tps.update_xaxes(
        title_text="Token/detik", range=[0, upper], row=1, col=col,
    )
fig_tps.update_yaxes(
    categoryorder="array", categoryarray=tps.index.tolist(),
    autorange="reversed",
)
fig_tps.update_layout(
    title=(
        "Kecepatan token, lebih tinggi lebih baik"
    ),
    template="plotly_white",
    height=max(440, 48 * len(tps) + 180),
    margin=dict(l=180, r=60, t=125, b=65),
)
fig_tps.show()

# 3. Quality
quality = chart_df.melt(
    id_vars="Model",
    value_vars=["HHEM", "BERTScore"],
    var_name="Metrik", value_name="Skor",
).dropna(subset=["Skor"])

quality_order = chart_df.sort_values(
    "HHEM", ascending=False, na_position="last",
)["Model"].tolist()

fig_quality = px.bar(
    quality,
    x="Skor", y="Model", color="Metrik",
    orientation="h", barmode="group",
    text_auto=".1%",
    category_orders={"Model": quality_order},
    title="Quality · mean skor lebih tinggi lebih baik",
    template="plotly_white",
)
fig_quality.update_xaxes(range=[0, 1], tickformat=".0%")
fig_quality.update_yaxes(title=None)
fig_quality.show()

# 4. Pareto: minimalkan waktu, maksimalkan HHEM
pareto = chart_df.dropna(subset=["Menit", "HHEM"]).copy()

pareto["Frontier"] = [
    not (
        (pareto["Menit"] <= row.Menit)
        & (pareto["HHEM"] >= row.HHEM)
        & (
            (pareto["Menit"] < row.Menit)
            | (pareto["HHEM"] > row.HHEM)
        )
    ).any()
    for row in pareto.itertuples(index=False)
]
pareto["Status"] = pareto["Frontier"].map({
    True: "Pareto frontier",
    False: "Di luar frontier",
})

frontier = (
    pareto.loc[pareto["Frontier"]]
    .sort_values("Menit")
    .drop_duplicates(["Menit", "HHEM"])
)

fig_pareto = px.scatter(
    pareto,
    x="Menit", y="HHEM",
    color="Status", text="Model",
    color_discrete_map={
        "Pareto frontier": "#2563eb",
        "Di luar frontier": "#9ca3af",
    },
    hover_data={"Menit": ":.2f", "HHEM": ":.2%"},
    labels={
        "Menit": "Total waktu summarization (menit)",
        "HHEM": "Mean HHEM consistency",
    },
    title="Speed vs quality · kiri atas lebih baik",
    template="plotly_white",
)
if len(frontier) > 1:
    fig_pareto.add_trace(go.Scatter(
        x=frontier["Menit"],
        y=frontier["HHEM"],
        mode="lines",
        line=dict(color="#2563eb", dash="dot"),
        showlegend=False,
        hoverinfo="skip",
    ))

label_positions = {
    "Gemma 4 E2B": "top right",
    "Ling 3.0 Tiny 8B": "bottom right",
    "Granite 4.2 3B": "middle left",
}
for trace in fig_pareto.data:
    if trace.mode == "markers+text":
        trace.update(
            textposition=[
                label_positions.get(name, "top center") for name in trace.text
            ],
            textfont_size=11,
            cliponaxis=False,
            marker_size=15 if trace.name == "Pareto frontier" else 10,
        )

fig_pareto.update_layout(
    height=560,
    margin=dict(l=80, r=60, t=125, b=80),
    legend=dict(
        title_text="", orientation="h", x=0, y=1.04, yanchor="bottom",
    ),
)
# Perbesar area skor tanpa menyembunyikan model yang skornya lebih rendah.
quality_lower = 0.65
if not pareto.empty:
    quality_lower = max(0, min(quality_lower, float(pareto["HHEM"].min()) - 0.05))
    padding = max(float(pareto["Menit"].max() - pareto["Menit"].min()) * 0.1, 2)
    fig_pareto.update_xaxes(range=[
        max(0, float(pareto["Menit"].min()) - padding),
        float(pareto["Menit"].max()) + padding,
    ])
fig_pareto.update_yaxes(
    tickformat=".0%", range=[quality_lower, 1.0],
    title="Mean HHEM (sumbu diperbesar)",
)
fig_pareto.show()